In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# 📊 Analyze Site Performance with Places Insights and BigQuery ML

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/googlemaps-samples/insights-samples/blob/main/places_insights/notebooks/analyze_site_performance/places_insights_analyze_site_performance_bigquery_ml.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2Fgooglemaps-samples%2Finsights-samples%2Fmain%2Fplaces_insights%2Fnotebooks%2Fanalyze_site_performance%2Fplaces_insights_analyze_site_performance_bigquery_ml.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/bigquery/import?url=https://github.com/googlemaps-samples/insights-samples/blob/main/places_insights/notebooks/analyze_site_performance/places_insights_analyze_site_performance_bigquery_ml.ipynb">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/bigquery/v1/32px.svg" alt="BigQuery Studio logo"><br> Open in BigQuery Studio
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/googlemaps-samples/insights-samples/blob/main/places_insights/notebooks/analyze_site_performance/places_insights_analyze_site_performance_bigquery_ml.ipynb">
      <img width="32px" src="https://upload.wikimedia.org/wikipedia/commons/9/91/Octicons-mark-github.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

### **Overview**

This notebook demonstrates a Geospatial Machine Learning workflow. We will combine internal operational metrics (synthetic store performance) with external environmental data (**Places Insights**) to diagnose the location factors that drive success.

By the end of this session, you will have a trained **Robust Linear Regression** model and an interactive **Prospecting Map** that scores every neighborhood in London based on its amenity profile.

### **Key Technologies**

*   **[Google Places Insights](https://developers.google.com/maps/documentation/placesinsights):** A BigQuery-native dataset providing aggregated counts of Places (POIs) without needing to query an API.
*   **[BigQuery ML](https://cloud.google.com/bigquery/docs/bqml-introduction):** Allows us to create, train, and deploy the machine learning model directly using standard SQL.
*   **[H3 Spatial Indexing](https://h3geo.org/):** We use H3 to divide the city into uniform cells for consistent scoring and visualization.
*   **[IPython Magics](https://googleapis.dev/python/bigquery-magics/latest/):** We use `%%bigquery` to write SQL directly in Colab cells.

### **The Workflow**

1.  **Data Ingestion:** We upload a synthetic dataset of 400 - 500 stores across **London** with varying performance scores.
2.  **Feature Engineering:** We use **Spatial Joins** (`ST_DWITHIN`) to count amenities (Gyms, Schools, Transit, etc.) within a 500m radius of every store.
3.  **Model Training:** We train a **Robust Linear Regression** model (`ML.ROBUST_SCALER`) to predict performance while handling geospatial outliers.
4.  **Evaluation:** We assess model accuracy using R² and Mean Absolute Error (MAE) on a holdout test set.
5.  **City-Wide Prospecting:** Instead of scoring a single site, we apply the model to the **entire London H3 Grid** (Resolution 8) to visualize performance hotspots across the city.
6.  **Clean Up:** We provide a final step to delete the dataset and all created tables/models from your Google Cloud project.

### **Prerequisites & Setup**

*   **Google Cloud Project:** You need a project with BigQuery enabled.
*   **Places Insights Access:** Your project must be subscribed to the [GB Places Insights dataset](https://developers.google.com/maps/documentation/placesinsights/cloud-setup) in BigQuery sharing.
*   **Google Maps Platform [API Key](https://developers.google.com/maps/get-started):** Required to render the final interactive map visualization. Enable the [**Maps JavaScript API**](https://developers.google.com/maps/documentation/javascript/get-api-key?setupProd=enable#enable-the-api) and [**Maps Tiles API**](https://developers.google.com/maps/documentation/tile/get-api-key?setupProd=enable#enable-the-api) on this key.
*   **Colab Secrets:** Please add the following to the **Secrets** tab (Key icon on the left):
    *   `GCP_PROJECT_ID`: Your Google Cloud Project ID.
    *   `GMP_API_KEY`: The Google Maps API Key you configured in the previous step.

In [ ]:
# @title 1a. Setup & Authentication
# @markdown Authenticate to Google Cloud, retrieve secrets, and initialize the BigQuery client.
# @markdown
# @markdown This cell creates a BigQuery dataset to use during this notebook, use this input to select the region (Default: US).

import sys
import random
import pandas as pd
import seaborn as sns
import pandas_gbq
from google.cloud import bigquery
import requests
import geopandas as gpd
import folium
import google.auth
import getpass

# --- 1. Global Configuration ---
DATASET_ID = "places_insights_site_perf_demo"
REGION = "US" # @param {type:"string"}
STORES_TABLE = f"{DATASET_ID}.store_performance"
FEATURES_TABLE = f"{DATASET_ID}.store_features"
MODEL_NAME = f"{DATASET_ID}.site_performance_model"

# --- 2. Authentication & Secrets ---
try:
    # ATTEMPT 1: Standard Consumer Colab
    from google.colab import auth, userdata

    GCP_PROJECT_ID = userdata.get('GCP_PROJECT_ID').strip()
    print(f"✅ Secrets retrieved for project: {GCP_PROJECT_ID}")

    GMP_API_KEY = userdata.get('GMP_API_KEY').strip()
    print("✅ GMP API Key retrieved.")

    print("🔄 Authenticating user...")
    auth.authenticate_user(project_id=GCP_PROJECT_ID)
    print("✅ User Authenticated.")

    # Initialize Client
    client = bigquery.Client(project=GCP_PROJECT_ID)

except (ImportError, Exception) as e:
    # ATTEMPT 2: Colab Enterprise / Local Jupyter
    print(f"ℹ️ Standard Colab setup skipped. Falling back to Enterprise/Local Auth...")

    # Retrieve environment credentials automatically
    credentials, GCP_PROJECT_ID = google.auth.default()
    print(f"✅ Authenticated via default credentials to Project: {GCP_PROJECT_ID}")

    # Securely prompt for the API key
    print("🔑 Please paste your Google Maps Platform API Key and press Enter:")
    GMP_API_KEY = getpass.getpass().strip()
    print("✅ GMP API Key securely captured.")

    # Initialize Client with enterprise credentials
    client = bigquery.Client(credentials=credentials, project=GCP_PROJECT_ID)

# --- 3. Initialize BigQuery Dataset ---
# This runs regardless of which auth method succeeded above
ds = bigquery.Dataset(f"{GCP_PROJECT_ID}.{DATASET_ID}")
ds.location = REGION
client.create_dataset(ds, exists_ok=True)
print(f"✅ Working dataset ready: {GCP_PROJECT_ID}.{DATASET_ID}")

In [ ]:
# @title 1b. Maps backend Initialization: Session, Copyright & Assets
# @markdown This cell manages the Maps API handshake. It performs the following steps:
# @markdown 1.  **Session Creation:** Authenticates and requests a "Roadmap" session for the target region.
# @markdown 2.  **Attribution Fetching:** Queries the API for the copyright text required for the configured viewport.
# @markdown 3.  **Asset Preparation:** Generates the HTML for the Google Maps logo overlay.

# --- 1. Create Google Maps Session ---
print("🗺️ Initializing Google Maps Session...")
session_url = f"https://tile.googleapis.com/v1/createSession?key={GMP_API_KEY}"
headers = {"Content-Type": "application/json"}
payload = {
    "mapType": "roadmap",
    "language": "en-GB",
    "region": "GB"
}

try:
    response = requests.post(session_url, json=payload, headers=headers)
    response.raise_for_status()
    session_token = response.json().get("session")
    print(f"✅ Session Token acquired.")
except Exception as e:
    raise RuntimeError(f"Failed to initialize Google Maps session: {e}")

# --- 2. Fetch Dynamic Attribution for London ---
# Center of our synthetic data area
LAT, LNG = 51.5074, -0.1278
ZOOM_LEVEL = 11
delta = 0.2

viewport_url = (
    f"https://tile.googleapis.com/tile/v1/viewport?key={GMP_API_KEY}"
    f"&session={session_token}"
    f"&zoom={ZOOM_LEVEL}"
    f"&north={LAT + delta}&south={LAT - delta}"
    f"&west={LNG - delta}&east={LNG + delta}"
)

try:
    vp_response = requests.get(viewport_url)
    vp_response.raise_for_status()
    google_attribution = vp_response.json().get('copyright', 'Map data © Google')
    print("✅ Attribution fetched.")
except Exception as e:
    print(f"⚠️ Warning: Could not fetch attribution ({e}). Defaulting.")
    google_attribution = "Map data © Google"

# --- 3. Construct Logo HTML ---
logo_url = "https://maps.gstatic.com/mapfiles/api-3/images/google_white3.png"
logo_html = f"""
    <div style="
        position: fixed;
        bottom: 35px;
        left: 10px;
        z-index: 9999;
        font-size: 0;
        pointer-events: none;
    ">
        <img src="{logo_url}" alt="Google Maps" style="height: 18px;">
    </div>
"""
print("✅ Logo HTML prepared.")

### 2. Import Data

In this step, we import the pre-generated dataset representing **store locations in London**.

This dataset contains:
*   `store_id`: Unique identifier.
*   `store_performance`: The synthetic performance score (0-100).
*   `geometry`: The geospatial location (Point).

We will upload the CSV locally and persist it to BigQuery to serve as the foundation for our model training.

In [ ]:
# @title 2a. Fetch Synthetic Data from GitHub
# @markdown We load the pre-generated `store_performance_london.csv` file directly from the public GitHub repository.
# @markdown
# @markdown *Curious how this synthetic dataset was created? Check out the [Data Generation Notebook](https://github.com/googlemaps-samples/insights-samples/blob/main/places_insights/notebooks/analyze_site_performance/places_insights_analyze_site_performance_data_generation.ipynb).*
import pandas as pd

github_url = "https://raw.githubusercontent.com/googlemaps-samples/insights-samples/main/places_insights/notebooks/analyze_site_performance/store_performance_london.csv"

print("⬇️ Fetching data from GitHub...")

# Read the CSV directly from the URL into a DataFrame
df_input = pd.read_csv(github_url)

print(f"✅ Successfully loaded {len(df_input)} rows.")
display(df_input.head())

In [ ]:
# @title 2b. Load Data to BigQuery
# @markdown We upload the DataFrame to the `STORES_TABLE` in BigQuery, casting the geometry column correctly.

# 1. Define Schema to ensure 'location' is parsed as GEOGRAPHY (not String)
table_schema = [
    {'name': 'store_id', 'type': 'STRING'},
    {'name': 'store_performance', 'type': 'FLOAT'},
    {'name': 'location', 'type': 'GEOGRAPHY'}, # Critical: Casts WKT string to GEOGRAPHY
]

# 2. Upload to BigQuery
print(f"☁️ Uploading data to `{STORES_TABLE}`...")

pandas_gbq.to_gbq(
    dataframe=df_input,
    destination_table=STORES_TABLE,
    project_id=GCP_PROJECT_ID,
    if_exists='replace',
    table_schema=table_schema,
    location=REGION
)

print(f"✅ Successfully loaded data to `{STORES_TABLE}`.")

In [ ]:
# @title 3. Feature Engineering (Spatial Join)
# @markdown We now bridge the gap between internal performance data and the external world using a **Spatial Join**.
# @markdown
# @markdown **The Logic:**
# @markdown 1.  **`ST_DWITHIN`:** For every store in our database, we look for Places within a **500-meter radius**.
# @markdown 2.  **`COUNTIF`:** We calculate density vectors (e.g., "How many gyms are nearby?") to serve as our model features ($X$).
# @markdown 3.  **Output:** The result is downloaded to the Python variable `df_features`.

%%bigquery df_features --project $GCP_PROJECT_ID --location $REGION

SELECT WITH AGGREGATION_THRESHOLD
    internal.store_id,
    internal.store_performance,

    -- Feature Engineering: count nearby POIs by type
    COUNTIF('gym' IN UNNEST(places.types)) AS gym_count,
    COUNTIF('restaurant' IN UNNEST(places.types)) AS restaurant_count,
    COUNTIF('school' IN UNNEST(places.types)) AS school_count,
    COUNTIF('transit_station' IN UNNEST(places.types)) AS transit_count,
    COUNTIF('clothing_store' IN UNNEST(places.types)) AS clothing_store_count

FROM
    `places_insights_site_perf_demo.store_performance` AS internal
JOIN
    `places_insights___gb.places` AS places
    ON ST_DWITHIN(internal.location, places.point, 500) -- 500m Radius
WHERE
    places.business_status = 'OPERATIONAL'
GROUP BY
    internal.store_id, internal.store_performance

In [ ]:
# @markdown Save the engineered features back to a permanent BQ table for training.

pandas_gbq.to_gbq( # type: ignore
    dataframe=df_features, # type: ignore
    destination_table=FEATURES_TABLE,
    project_id=GCP_PROJECT_ID,
    if_exists='replace',
    location=REGION
)

print(f"✅ Training data saved to `{FEATURES_TABLE}`")
display(df_features.head()) # type: ignore

In [ ]:
# @title **Exploratory Data Analysis: Feature Correlations**
# @markdown We use a **Pairplot** to visualize how each feature interacts with the target variable (`store_performance`).
# @markdown
# @markdown **Key Observations:**
# @markdown *   **Linearity:** Look at the top row. You can see a clear positive linear trend between features like `restaurant_count` and `store_performance`. This confirms that a **Linear Regression** model is the right choice for this data.
# @markdown *   **Distributions:** The diagonal histograms show that our amenity counts are "right-skewed" (mostly low numbers with a few high-density hubs), which is typical for geospatial data.

import matplotlib.pyplot as plt

input_features = ['store_performance', 'gym_count', 'restaurant_count', 'school_count', 'transit_count', 'clothing_store_count']
g = sns.pairplot(df_features[input_features], plot_kws={"s": 3, 'alpha': 0.6}, diag_kws={'color': 'crimson'}, height=1.8)  # type: ignore
g.set(xlim=(0, 100), ylim=(0, 100))

plt.show()

In [ ]:
# @title 4. Train the Linear Regression Model
# @markdown We now train a **Linear Regression** model to predict store performance.
# @markdown
# @markdown **Key Model Configuration:**
# @markdown *   **`ML.ROBUST_SCALER`:** We use this within the `TRANSFORM` clause. Unlike standard scaling (Mean/StdDev), robust scaling uses the **Median** and **IQR**. This is critical for geospatial data, where a single location with 500 restaurants (an outlier) could otherwise skew the entire model.
# @markdown *   **`AUTO_SPLIT`:** We let BigQuery automatically reserve ~20% of the data for evaluation. This makes sure we test the model on data it has never seen before.
# @markdown *   **`NORMAL_EQUATION`:** Since our dataset is small, we use the exact mathematical solution rather than an iterative approximation.
# @markdown *   **Outlier Removal:** We filter out stores with `performance > 75` to focus the model on predicting the mechanics of "typical" or "developing" sites, rather than established outliers.

%%bigquery --project $GCP_PROJECT_ID --location $REGION

CREATE OR REPLACE MODEL `places_insights_site_perf_demo.site_performance_model`
TRANSFORM(
  store_performance,
  -- Feature Engineering inside the model artifact
  -- These stats are calculated on the TRAINING split only
  ML.ROBUST_SCALER(gym_count) OVER() AS scaled_gym_count,
  ML.ROBUST_SCALER(restaurant_count) OVER() AS scaled_restaurant_count,
  ML.ROBUST_SCALER(school_count) OVER() AS scaled_school_count,
  ML.ROBUST_SCALER(transit_count) OVER() AS scaled_transit_count,
  ML.ROBUST_SCALER(clothing_store_count) OVER() AS scaled_clothing_store_count
)
OPTIONS(
    model_type = 'LINEAR_REG',
    input_label_cols = ['store_performance'],

    -- OPTIMIZATION PARAMETERS
    optimize_strategy = 'NORMAL_EQUATION', -- Exact mathematical solution (fast for small data)
    data_split_method = 'AUTO_SPLIT',      -- Automatically reserves ~20% for evaluation

    -- DIAGNOSTICS
    enable_global_explain = TRUE -- Essential to see feature importance
)
AS
SELECT
    gym_count,
    restaurant_count,
    school_count,
    transit_count,
    clothing_store_count,
    store_performance
FROM
    `places_insights_site_perf_demo.store_features`
WHERE
    store_performance < 75

In [ ]:
# @title 5. Evaluate model performance
# @markdown We use `ML.EVALUATE` to test the model against the unseen "Holdout" data (the ~20% reserved automatically during training).
# @markdown The results (MAE, R2, etc.) are downloaded to the `df_eval` DataFrame for inspection in the next step.

%%bigquery df_eval --project $GCP_PROJECT_ID --location $REGION

SELECT *
FROM ML.EVALUATE(MODEL `places_insights_site_perf_demo.site_performance_model`)

In [ ]:
# @markdown ### **Interpretation of Results**
# @markdown *   **R2 Score:** Measures how well the amenities explain the performance. A score close to **1.0** indicates a perfect fit. Since our data is synthetic and linear, we expect a very high score here (> 0.9).
# @markdown *   **Mean Absolute Error (MAE):** The average "miss" in points. For example, an MAE of **1.5** means the model's prediction is typically within +/- 1.5 points of the actual score.

print(f"R2 Score: {df_eval['r2_score'][0]:.4f}") # type: ignore
print(f"Mean Absolute Error: {df_eval['mean_absolute_error'][0]:.4f}") # type: ignore
display(df_eval) # type: ignore

In [ ]:
# @title 6. Score London by H3 Cell (Native Places Insights)
# @markdown We now apply our trained model to London using the native Places Insights H3 function.
# @markdown
# @markdown **The Approach:**
# @markdown 1.  **H3 Indexing & Counting:** We use `PLACES_COUNT_PER_H3` to get pre-aggregated counts of amenities within a 25km radius of central London.
# @markdown 2.  **Pivoting:** Because the function returns one row per amenity type, we use `UNION ALL` and group the results to create the feature columns (`gym_count`, `restaurant_count`, etc.).
# @markdown 3.  **Batch Prediction:** We feed these "Grid Features" into `ML.PREDICT` to generate a `predicted_store_performance` score for every cell.

%%bigquery df_h3_predictions --project $GCP_PROJECT_ID --location $REGION

WITH combined_counts AS (
    -- Gyms
    SELECT h3_cell_index, geography, count, 'gym' AS type
    FROM `places_insights___gb.PLACES_COUNT_PER_H3`(
        JSON_OBJECT(
            'geography', ST_BUFFER(ST_GEOGPOINT(-0.1278, 51.5074), 25000), -- 25km radius around London
            'h3_resolution', 8,
            'business_status', ['OPERATIONAL'],
            'types', ['gym']
        )
    )
    UNION ALL
    -- Restaurants
    SELECT h3_cell_index, geography, count, 'restaurant' AS type
    FROM `places_insights___gb.PLACES_COUNT_PER_H3`(
        JSON_OBJECT(
            'geography', ST_BUFFER(ST_GEOGPOINT(-0.1278, 51.5074), 25000),
            'h3_resolution', 8,
            'business_status', ['OPERATIONAL'],
            'types', ['restaurant']
        )
    )
    UNION ALL
    -- Schools
    SELECT h3_cell_index, geography, count, 'school' AS type
    FROM `places_insights___gb.PLACES_COUNT_PER_H3`(
        JSON_OBJECT(
            'geography', ST_BUFFER(ST_GEOGPOINT(-0.1278, 51.5074), 25000),
            'h3_resolution', 8,
            'business_status', ['OPERATIONAL'],
            'types', ['school']
        )
    )
    UNION ALL
    -- Transit Stations
    SELECT h3_cell_index, geography, count, 'transit_station' AS type
    FROM `places_insights___gb.PLACES_COUNT_PER_H3`(
        JSON_OBJECT(
            'geography', ST_BUFFER(ST_GEOGPOINT(-0.1278, 51.5074), 25000),
            'h3_resolution', 8,
            'business_status', ['OPERATIONAL'],
            'types', ['transit_station']
        )
    )
    UNION ALL
    -- Clothing Stores
    SELECT h3_cell_index, geography, count, 'clothing_store' AS type
    FROM `places_insights___gb.PLACES_COUNT_PER_H3`(
        JSON_OBJECT(
            'geography', ST_BUFFER(ST_GEOGPOINT(-0.1278, 51.5074), 25000),
            'h3_resolution', 8,
            'business_status', ['OPERATIONAL'],
            'types', ['clothing_store']
        )
    )
),
aggregated_features AS (
    -- Pivot the stacked rows back into standard feature columns for the ML Model
    SELECT
        h3_cell_index AS h3_index,
        ANY_VALUE(geography) AS h3_geography,
        SUM(IF(type = 'gym', count, 0)) AS gym_count,
        SUM(IF(type = 'restaurant', count, 0)) AS restaurant_count,
        SUM(IF(type = 'school', count, 0)) AS school_count,
        SUM(IF(type = 'transit_station', count, 0)) AS transit_count,
        SUM(IF(type = 'clothing_store', count, 0)) AS clothing_store_count
    FROM
        combined_counts
    GROUP BY
        h3_cell_index
)

-- Feed the pivoted features into the model
SELECT
    h3_index,
    predicted_store_performance,
    h3_geography,
    gym_count,
    restaurant_count
FROM
    ML.PREDICT(MODEL `places_insights_site_perf_demo.site_performance_model`,
      (SELECT * FROM aggregated_features)
    )
ORDER BY
    predicted_store_performance DESC;

In [ ]:
# @title 7. Display H3 Prospecting Map
# @markdown We render the H3 grid as a choropleth layer.
# @markdown *   **Yellow Areas:** High predicted performance (Hotspots).
# @markdown *   **Purple Areas:** Low predicted performance (Coldspots).
# @markdown *   **Interactive:** Hover over any cell to see the underlying amenity counts driving the score.

import geopandas as gpd
from folium import Element
from shapely import wkt

# --- 1. Prepare Data ---
# Explicitly convert the WKT strings from BigQuery into Shapely Geometry objects
if isinstance(df_h3_predictions['h3_geography'].iloc[0], str):
    df_h3_predictions['h3_geography'] = df_h3_predictions['h3_geography'].apply(wkt.loads)

# Create GeoDataFrame
gdf_h3 = gpd.GeoDataFrame(df_h3_predictions, geometry='h3_geography')

# --- 2. Construct Tiles URL ---
tiles_url = f"https://tile.googleapis.com/v1/2dtiles/{{z}}/{{x}}/{{y}}?session={session_token}&key={GMP_API_KEY}"

# --- 3. Initialize Map ---
m = folium.Map(
    location=[51.5074, -0.1278],
    zoom_start=11,
    tiles=tiles_url,
    attr=google_attribution,
    name="Google Maps",
    control_scale=True,
    prefer_canvas=True
)

# --- 4. Add Google Logo (Bottom Left) ---
m.get_root().html.add_child(Element(logo_html))

# --- 5. Add Custom Legend (Bottom Right) ---
legend_html = """
<div style="
    position: fixed;
    bottom: 35px;
    right: 10px;
    width: 160px;
    height: 80px;
    z-index: 9999;
    background-color: rgba(255, 255, 255, 0.9);
    border-radius: 4px;
    box-shadow: 0 1px 4px rgba(0,0,0,0.3);
    padding: 10px;
    font-family: Roboto, Arial, sans-serif;
    font-size: 12px;
">
    <strong style="font-size:13px;">Predicted Score</strong>
    <div style="
        width: 100%;
        height: 12px;
        /* Viridis Color Scale: Purple -> Teal -> Green -> Yellow */
        background: linear-gradient(to right, #440154, #3b528b, #21918c, #5ec962, #fde725);
        margin-top: 8px;
        margin-bottom: 4px;
    "></div>
    <div style="display: flex; justify-content: space-between; color: #555;">
        <span>Low (~20)</span>
        <span>High (~80)</span>
    </div>
</div>
"""
m.get_root().html.add_child(Element(legend_html))

# --- 6. Overlay H3 Grid ---
gdf_h3.explore(
    m=m,
    column='predicted_store_performance',
    cmap='viridis',
    vmin=20,
    vmax=80,
    # Style Keywords for Polygons (remove borders for smoother look)
    style_kwds={'stroke': False, 'fillOpacity': 0.6},
    tooltip=[
        'h3_index',
        'predicted_store_performance',
        'gym_count',
        'restaurant_count'
        # Note: 'transit_count' removed because it wasn't selected in the SQL query
    ],
    name="Prospecting Heatmap"
)

# Add layer control to toggle data on/off
folium.LayerControl().add_to(m)

display(m)

In [ ]:
# @title 8. Clean Up Resources
# @markdown This cell deletes the demo dataset (`places_insights_site_perf_demo`) and all tables within it.
# @markdown
# @markdown **You will be prompted to confirm before deletion.**

from google.cloud.exceptions import NotFound

# Validation
print(f"⚠️ WARNING: You are about to DELETE the dataset: `{DATASET_ID}`")
print(f"   Project: `{GCP_PROJECT_ID}`")
print("   This action cannot be undone.")

# Interactive Input
confirmation = input("Type 'yes' to proceed with deletion: ").strip().lower()

if confirmation == 'yes':
    print(f"\n🗑️ Deleting dataset: {DATASET_ID}...")
    try:
        # delete_contents=True removes tables inside the dataset
        # not_found_ok=True prevents errors if the dataset is already gone
        client.delete_dataset(DATASET_ID, delete_contents=True, not_found_ok=True)
        print(f"✅ Successfully deleted dataset '{DATASET_ID}' and all contents.")
    except Exception as e:
        print(f"❌ Error deleting dataset: {e}")
else:
    print(f"\n🛑 Operation cancelled. Dataset `{DATASET_ID}` was NOT deleted.")